1. Load the Data in PySpark

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('PySpark Assessment').getOrCreate()

In [3]:
from google.colab import files
uploaded = files.upload()

Saving machine_configurations.json to machine_configurations.json
Saving machine_master.csv to machine_master.csv
Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv
Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Saving plant_master.csv to plant_master.csv


In [4]:
df = spark.read.csv(
 "manufacturing_production_120k.csv",
 header=True,
 inferSchema=True
)


In [8]:
df.show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [9]:
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



In [10]:
print(df.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [11]:
df.count()

120000

In [19]:
df.select('production_id','plant_id','machine_id','product_family','units_produced').show(20)

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|    PR0000001|    PL03|     MC011|        Motors|           993|
|    PR0000002|    PL04|     MC019|   Controllers|           836|
|    PR0000003|    PL01|     MC002|        Motors|           380|
|    PR0000004|    PL03|     MC013|         Pumps|           720|
|    PR0000005|    PL04|     MC020|         Pumps|           296|
|    PR0000006|    PL01|     MC002|        Motors|           795|
|    PR0000007|    PL02|     MC007|        Valves|           955|
|    PR0000008|    PL02|     MC009|        Valves|           895|
|    PR0000009|    PL03|     MC015|   Controllers|           289|
|    PR0000010|    PL02|     MC006|   Controllers|           527|
|    PR0000011|    PL03|     MC012|         Pumps|           512|
|    PR0000012|    PL06|     MC030|   Controllers|           968|
|    PR000

In [20]:
for field in df.schema.fields:
  print(field.name,field.dataType)

production_id StringType()
production_date DateType()
plant_id StringType()
machine_id StringType()
operator_id StringType()
shift StringType()
product_code StringType()
product_family StringType()
units_planned IntegerType()
units_produced IntegerType()
defective_units IntegerType()
downtime_minutes IntegerType()
energy_kwh DoubleType()
temperature_c DoubleType()
vibration_mm_s DoubleType()
quality_status StringType()
maintenance_flag StringType()
supervisor StringType()


In [22]:
df.sample(fraction=0.001).limit(10).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0001903|     2026-03-24|    PL01|     MC001|     OP0191|Morning|     VLV-220|        Valves|          358|           363|              4|              28|    146.94|        65.57|          3.05|          Pass|              No|    Bharat|
|    PR0002049|     

2. Creating DataFrames

In [23]:
plant_rows = [
    ("PL01", "Hyderabad Plant", "Hyderabad"),
    ("PL02", "Pune Plant", "Pune"),
    ("PL03", "Chennai Plant", "Chennai")
]

plant_columns = ["plant_id", "plant_name", "city"]
small_plant_df = spark.createDataFrame(plant_rows, plant_columns)
small_plant_df.show()

+--------+---------------+---------+
|plant_id|     plant_name|     city|
+--------+---------------+---------+
|    PL01|Hyderabad Plant|Hyderabad|
|    PL02|     Pune Plant|     Pune|
|    PL03|  Chennai Plant|  Chennai|
+--------+---------------+---------+



In [24]:
small_plant_df.printSchema()

root
 |-- plant_id: string (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



In [25]:
from pyspark.sql.types import StructType, StructField, StringType
plant_schema = StructType([
    StructField("plant_id", StringType(), False),
    StructField("plant_name", StringType(), False),
    StructField("city", StringType(), False)
])

small_plant_schema_df = spark.createDataFrame(plant_rows, plant_schema)
small_plant_schema_df.show()

+--------+---------------+---------+
|plant_id|     plant_name|     city|
+--------+---------------+---------+
|    PL01|Hyderabad Plant|Hyderabad|
|    PL02|     Pune Plant|     Pune|
|    PL03|  Chennai Plant|  Chennai|
+--------+---------------+---------+



In [27]:
from pyspark.sql.functions import lit
plant_with_country = small_plant_df.withColumn("country", lit("India"))
plant_with_country.show()

+--------+---------------+---------+-------+
|plant_id|     plant_name|     city|country|
+--------+---------------+---------+-------+
|    PL01|Hyderabad Plant|Hyderabad|  India|
|    PL02|     Pune Plant|     Pune|  India|
|    PL03|  Chennai Plant|  Chennai|  India|
+--------+---------------+---------+-------+



3. Pandas DataFrame

In [31]:
import pandas as pd
pdf = pd.read_csv("manufacturing_production_120k.csv")
display(pdf.head())

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [32]:
display(pdf.tail(5))

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
119995,PR0119996,2026-07-13,PL05,MC022,OP0193,Evening,MTR-200,Motors,435,399,1,73.0,126.07,62.79,5.73,Rework,No,Bharat
119996,PR0119997,2026-04-26,PL05,MC024,OP0124,Evening,CTL-130,Controllers,813,777,8,49.0,318.22,63.05,3.53,Pass,No,Irfan
119997,PR0119998,2026-02-01,PL06,MC027,OP0051,Evening,VLV-120,Valves,650,597,13,0.0,269.46,60.52,3.81,Pass,No,Bharat
119998,PR0119999,2026-01-17,PL01,MC001,OP0452,Evening,MTR-300,Motors,887,899,22,21.0,184.62,64.89,5.20,Pass,Yes,Farah
119999,PR0120000,2026-08-16,PL02,MC007,OP0420,Morning,MTR-200,Motors,547,477,4,43.0,107.95,62.50,3.97,Pass,No,Farah


In [35]:
print(pdf.shape)

(120000, 18)


In [38]:
pdf.columns.tolist()

['production_id',
 'production_date',
 'plant_id',
 'machine_id',
 'operator_id',
 'shift',
 'product_code',
 'product_family',
 'units_planned',
 'units_produced',
 'defective_units',
 'downtime_minutes',
 'energy_kwh',
 'temperature_c',
 'vibration_mm_s',
 'quality_status',
 'maintenance_flag',
 'supervisor']

In [39]:
pdf.dtypes

,0
production_id,object
production_date,object
plant_id,object
machine_id,object
operator_id,object
shift,object
product_code,object
product_family,object
units_planned,int64
units_produced,int64


In [40]:
pdf[["plant_id", "machine_id", "units_produced"]].head()

,plant_id,machine_id,units_produced
0,PL03,MC011,993
1,PL04,MC019,836
2,PL01,MC002,380
3,PL03,MC013,720
4,PL04,MC020,296


In [41]:
pdf_renamed = pdf.rename(columns={"units_produced": "production_output"})
pdf_renamed.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,production_output,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [42]:
pdf[pdf["quality_status"] == "Reject"].head(10)

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
7,PR0000008,2026-08-17,PL02,MC009,OP0106,Night,VLV-320,Valves,862,895,51,77.0,266.06,57.10,1.34,Reject,No,Irfan
9,PR0000010,2026-02-13,PL02,MC006,OP0593,Evening,CTL-230,Controllers,532,527,26,64.0,112.17,60.96,3.48,Reject,No,Gopal
21,PR0000022,2026-03-03,PL02,MC009,OP0294,Evening,MTR-300,Motors,623,560,23,14.0,217.69,55.23,3.28,Reject,Yes,Farah
22,PR0000023,2026-10-15,PL04,MC019,OP0187,Morning,VLV-120,Valves,1031,942,29,42.0,268.18,68.90,1.92,Reject,No,Bharat
23,PR0000024,2026-06-07,PL03,MC013,OP0456,Morning,MTR-100,Motors,495,456,3,14.0,222.03,67.44,3.71,Reject,No,Harini
28,PR0000029,2026-04-30,PL05,MC023,OP0339,Night,VLV-120,Valves,987,1010,25,53.0,199.38,65.76,4.48,Reject,No,Harini
30,PR0000031,2026-01-04,PL01,MC003,OP0524,Evening,PMP-210,Pumps,762,776,49,28.0,238.12,66.80,3.09,Reject,No,Bharat
45,PR0000046,2026-04-02,PL06,MC027,OP0484,Morning,VLV-120,Valves,847,778,19,145.0,381.88,63.42,2.94,Reject,No,Anita


In [43]:
pdf[pdf["temperature_c"] > 75].head(10)

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
14,PR0000015,2026-03-15,PL02,MC007,OP0238,Morning,MTR-300,Motors,1086,1042,4,172.0,239.64,76.39,2.81,Pass,No,Deepa
16,PR0000017,2026-08-04,PL03,MC011,OP0533,Morning,CTL-330,Controllers,1152,963,28,91.0,347.34,75.09,4.00,Pass,Yes,Gopal
19,PR0000020,2026-02-18,PL03,MC014,OP0563,Morning,PMP-310,Pumps,646,573,13,64.0,277.73,76.92,2.35,Pass,No,Irfan
26,PR0000027,2026-01-03,PL05,MC023,OP0190,Morning,PMP-210,Pumps,358,310,6,46.0,78.29,75.42,3.72,Rework,No,Anita
54,PR0000055,2026-07-31,PL01,MC005,OP0212,Morning,CTL-230,Controllers,499,478,10,32.0,112.60,75.10,3.38,Pass,No,Farah
55,PR0000056,2026-09-01,PL06,MC029,OP0239,Night,VLV-320,Valves,1102,970,62,55.0,450.64,79.15,3.88,Rework,Yes,Deepa
63,PR0000064,2026-10-21,PL05,MC022,OP0267,Evening,CTL-230,Controllers,805,794,16,38.0,189.94,76.15,3.55,Pass,No,Anita
69,PR0000070,2026-08-29,PL04,MC017,OP0460,Morning,CTL-230,Controllers,614,625,33,42.0,200.11,76.02,5.20,Reject,Yes,Farah
86,PR0000087,2026-01-02,PL03,MC014,OP0488,Evening,VLV-120,Valves,700,608,33,40.0,251.91,75.41,2.47,Reject,No,Bharat
109,PR0000110,2026-07-07,PL02,MC008,OP0598,Evening,CTL-130,Controllers,628,549,5,65.0,213.65,77.27,2.73,Rework,No,Deepa


In [44]:
night_hyderabad = pdf[
    (pdf["shift"] == "Night") & (pdf['plant_id'] == "PL01")
]
night_hyderabad.head(10)

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
29,PR0000030,2026-09-18,PL01,MC004,OP0266,Night,VLV-320,Valves,358,359,1,19.0,107.98,72.00,1.73,Pass,No,Bharat
38,PR0000039,2026-07-17,PL01,MC002,OP0220,Night,VLV-120,Valves,260,265,0,2.0,127.18,51.87,3.41,Pass,No,Irfan
62,PR0000063,2026-02-18,PL01,MC002,OP0390,Night,VLV-120,Valves,656,665,0,44.0,311.80,58.34,2.00,Pass,No,Deepa
79,PR0000080,2026-06-10,PL01,MC003,OP0214,Night,CTL-130,Controllers,948,821,49,20.0,188.65,62.68,2.85,Reject,No,Anita
145,PR0000146,2026-08-19,PL01,MC005,OP0088,Night,VLV-120,Valves,1156,969,0,9.0,501.65,67.41,5.76,Rework,No,Bharat
159,PR0000160,2026-10-11,PL01,MC003,OP0182,Night,CTL-330,Controllers,593,532,15,0.0,119.03,62.59,5.91,Reject,Yes,Farah
201,PR0000202,2026-09-09,PL01,MC003,OP0161,Night,MTR-300,Motors,596,562,0,27.0,208.91,69.23,5.59,Rework,No,Deepa
232,PR0000233,2026-04-20,PL01,MC002,OP0437,Night,MTR-300,Motors,1084,947,53,59.0,404.27,69.34,2.31,Reject,No,Charan
242,PR0000243,2026-04-09,PL01,MC001,OP0151,Night,MTR-200,Motors,535,464,10,31.0,197.65,52.66,2.48,Rework,No,Farah
249,PR0000250,2026-03-13,PL01,MC003,OP0394,Night,CTL-130,Controllers,924,832,60,0.0,242.28,65.18,3.41,Reject,No,Anita


4. Dask DataFrame

In [54]:
import dask.dataframe as dd
ddf = dd.read_csv(
 "manufacturing_production_120k.csv", assume_missing=True
)

In [55]:
print(ddf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors         1060.0           993.0             30.0   
1      CTL-330    Controllers          809.0           836.0              0.0   
2      MTR-300         Motors          442.0           380.0             17.0   
3      PMP-210          Pumps          787.0           720.0              7.0   
4      PMP-110          Pumps          344.0           296.0              7.0   

   downtime_minutes  energy_kwh  temperature_c  vibratio

In [56]:
print(ddf.npartitions)

1


In [57]:
ddf[["plant_id", "product_family", "units_produced"]].head(10)

,plant_id,product_family,units_produced
0,PL03,Motors,993.0
1,PL04,Controllers,836.0
2,PL01,Motors,380.0
3,PL03,Pumps,720.0
4,PL04,Pumps,296.0
5,PL01,Motors,795.0
6,PL02,Valves,955.0
7,PL02,Valves,895.0
8,PL03,Controllers,289.0
9,PL02,Controllers,527.0


In [58]:
rejected_ddf = ddf[ddf["quality_status"] == "Reject"]
display(rejected_ddf.head(10))

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809.0,836.0,0.0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442.0,380.0,17.0,15.0,183.34,54.13,4.66,Reject,No,Deepa
7,PR0000008,2026-08-17,PL02,MC009,OP0106,Night,VLV-320,Valves,862.0,895.0,51.0,77.0,266.06,57.10,1.34,Reject,No,Irfan
9,PR0000010,2026-02-13,PL02,MC006,OP0593,Evening,CTL-230,Controllers,532.0,527.0,26.0,64.0,112.17,60.96,3.48,Reject,No,Gopal
21,PR0000022,2026-03-03,PL02,MC009,OP0294,Evening,MTR-300,Motors,623.0,560.0,23.0,14.0,217.69,55.23,3.28,Reject,Yes,Farah
22,PR0000023,2026-10-15,PL04,MC019,OP0187,Morning,VLV-120,Valves,1031.0,942.0,29.0,42.0,268.18,68.90,1.92,Reject,No,Bharat
23,PR0000024,2026-06-07,PL03,MC013,OP0456,Morning,MTR-100,Motors,495.0,456.0,3.0,14.0,222.03,67.44,3.71,Reject,No,Harini
28,PR0000029,2026-04-30,PL05,MC023,OP0339,Night,VLV-120,Valves,987.0,1010.0,25.0,53.0,199.38,65.76,4.48,Reject,No,Harini
30,PR0000031,2026-01-04,PL01,MC003,OP0524,Evening,PMP-210,Pumps,762.0,776.0,49.0,28.0,238.12,66.80,3.09,Reject,No,Bharat
45,PR0000046,2026-04-02,PL06,MC027,OP0484,Morning,VLV-120,Valves,847.0,778.0,19.0,145.0,381.88,63.42,2.94,Reject,No,Anita


In [59]:
avg_by_plant_dask = ddf.groupby("plant_id")["units_produced"].mean().compute().reset_index()
display(avg_by_plant_dask)


,plant_id,units_produced
0,PL01,672.998304
1,PL02,672.781326
2,PL03,675.665503
3,PL04,673.835101
4,PL05,673.124994
5,PL06,678.006884


In [60]:
defects_by_family_dask = ddf.groupby("product_family")["defective_units"].sum().compute().reset_index()
display(defects_by_family_dask)

,product_family,defective_units
0,Controllers,507781.0
1,Motors,505287.0
2,Pumps,507864.0
3,Valves,509312.0


In [61]:
pandas_avg = pdf.groupby("plant_id", as_index=False)["units_produced"].mean()
print(pandas_avg)

  plant_id  units_produced
0     PL01      672.998304
1     PL02      672.781326
2     PL03      675.665503
3     PL04      673.835101
4     PL05      673.124994
5     PL06      678.006884


In [62]:
dask_avg = ddf.groupby("plant_id")["units_produced"].mean().compute().reset_index()
display(dask_avg)

,plant_id,units_produced
0,PL01,672.998304
1,PL02,672.781326
2,PL03,675.665503
3,PL04,673.835101
4,PL05,673.124994
5,PL06,678.006884


5. Manupulating DataFrames

In [64]:
from pyspark.sql.functions import when, col
df.withColumn(
    "production_efficiency",
    when(
        col("units_planned") != 0,
        col("units_produced") / col("units_planned") * 100
    )
).select("production_id", "units_planned", "units_produced", "production_efficiency").show(10)

+-------------+-------------+--------------+---------------------+
|production_id|units_planned|units_produced|production_efficiency|
+-------------+-------------+--------------+---------------------+
|    PR0000001|         1060|           993|    93.67924528301886|
|    PR0000002|          809|           836|   103.33745364647713|
|    PR0000003|          442|           380|    85.97285067873304|
|    PR0000004|          787|           720|     91.4866581956798|
|    PR0000005|          344|           296|    86.04651162790698|
|    PR0000006|          958|           795|    82.98538622129436|
|    PR0000007|         1165|           955|    81.97424892703863|
|    PR0000008|          862|           895|   103.82830626450117|
|    PR0000009|          313|           289|    92.33226837060703|
|    PR0000010|          532|           527|    99.06015037593986|
+-------------+-------------+--------------+---------------------+
only showing top 10 rows


In [65]:
from pyspark.sql import functions as F

df.withColumn(
    "good_units",
    F.col("units_produced") - F.col("defective_units")
).select("production_id", "units_produced", "defective_units", "good_units").show(10)

+-------------+--------------+---------------+----------+
|production_id|units_produced|defective_units|good_units|
+-------------+--------------+---------------+----------+
|    PR0000001|           993|             30|       963|
|    PR0000002|           836|              0|       836|
|    PR0000003|           380|             17|       363|
|    PR0000004|           720|              7|       713|
|    PR0000005|           296|              7|       289|
|    PR0000006|           795|             50|       745|
|    PR0000007|           955|             10|       945|
|    PR0000008|           895|             51|       844|
|    PR0000009|           289|              8|       281|
|    PR0000010|           527|             26|       501|
+-------------+--------------+---------------+----------+
only showing top 10 rows


In [66]:
df.withColumn("country", F.lit("India")).select("production_id", "plant_id", "country").show(10)

+-------------+--------+-------+
|production_id|plant_id|country|
+-------------+--------+-------+
|    PR0000001|    PL03|  India|
|    PR0000002|    PL04|  India|
|    PR0000003|    PL01|  India|
|    PR0000004|    PL03|  India|
|    PR0000005|    PL04|  India|
|    PR0000006|    PL01|  India|
|    PR0000007|    PL02|  India|
|    PR0000008|    PL02|  India|
|    PR0000009|    PL03|  India|
|    PR0000010|    PL02|  India|
+-------------+--------+-------+
only showing top 10 rows


In [67]:
df_renamed = df.withColumnRenamed("energy_kwh", "energy_consumed")
df_renamed.select("production_id", "energy_consumed").show(10)

+-------------+---------------+
|production_id|energy_consumed|
+-------------+---------------+
|    PR0000001|         388.11|
|    PR0000002|         398.58|
|    PR0000003|         183.34|
|    PR0000004|         217.61|
|    PR0000005|         115.47|
|    PR0000006|         290.26|
|    PR0000007|         355.56|
|    PR0000008|         266.06|
|    PR0000009|          87.62|
|    PR0000010|         112.17|
+-------------+---------------+
only showing top 10 rows


In [68]:
df_double = df.withColumn("units_produced", F.col("units_produced").cast("double"))
df_double.select("production_id", "units_produced").printSchema()

root
 |-- production_id: string (nullable = true)
 |-- units_produced: double (nullable = true)



In [69]:
df.withColumn(
    "defect_percentage",
    F.when(
        F.col("units_produced") != 0,
        F.col("defective_units") / F.col("units_produced") * 100
    )
).select("production_id", "units_produced", "defective_units", "defect_percentage").show(10)

+-------------+--------------+---------------+------------------+
|production_id|units_produced|defective_units| defect_percentage|
+-------------+--------------+---------------+------------------+
|    PR0000001|           993|             30|3.0211480362537766|
|    PR0000002|           836|              0|               0.0|
|    PR0000003|           380|             17| 4.473684210526316|
|    PR0000004|           720|              7|0.9722222222222222|
|    PR0000005|           296|              7| 2.364864864864865|
|    PR0000006|           795|             50| 6.289308176100629|
|    PR0000007|           955|             10|1.0471204188481675|
|    PR0000008|           895|             51| 5.698324022346369|
|    PR0000009|           289|              8| 2.768166089965398|
|    PR0000010|           527|             26| 4.933586337760911|
+-------------+--------------+---------------+------------------+
only showing top 10 rows


In [70]:
df.withColumn(
    "energy_per_unit",
    F.when(
        F.col("units_produced") != 0,
        F.col("energy_kwh") / F.col("units_produced")
    )
).select("production_id", "energy_kwh", "units_produced", "energy_per_unit").show(10)

+-------------+----------+--------------+-------------------+
|production_id|energy_kwh|units_produced|    energy_per_unit|
+-------------+----------+--------------+-------------------+
|    PR0000001|    388.11|           993|0.39084592145015107|
|    PR0000002|    398.58|           836| 0.4767703349282296|
|    PR0000003|    183.34|           380|0.48247368421052633|
|    PR0000004|    217.61|           720|0.30223611111111115|
|    PR0000005|    115.47|           296| 0.3901013513513513|
|    PR0000006|    290.26|           795| 0.3651069182389937|
|    PR0000007|    355.56|           955|0.37231413612565445|
|    PR0000008|    266.06|           895| 0.2972737430167598|
|    PR0000009|     87.62|           289| 0.3031833910034602|
|    PR0000010|    112.17|           527| 0.2128462998102467|
+-------------+----------+--------------+-------------------+
only showing top 10 rows


In [71]:
df.select(
    "production_id",
    "units_planned",
    "units_produced",
    "defective_units",
    "energy_kwh"
).show(10)

+-------------+-------------+--------------+---------------+----------+
|production_id|units_planned|units_produced|defective_units|energy_kwh|
+-------------+-------------+--------------+---------------+----------+
|    PR0000001|         1060|           993|             30|    388.11|
|    PR0000002|          809|           836|              0|    398.58|
|    PR0000003|          442|           380|             17|    183.34|
|    PR0000004|          787|           720|              7|    217.61|
|    PR0000005|          344|           296|              7|    115.47|
|    PR0000006|          958|           795|             50|    290.26|
|    PR0000007|         1165|           955|             10|    355.56|
|    PR0000008|          862|           895|             51|    266.06|
|    PR0000009|          313|           289|              8|     87.62|
|    PR0000010|          532|           527|             26|    112.17|
+-------------+-------------+--------------+---------------+----

6. Dropping Data

In [72]:
df.drop("supervisor").show(5)


+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|M

In [73]:
df.drop("supervisor", "maintenance_flag").show(5)


+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|Morning|     CTL-330|   Controllers|          809|           836|    

In [74]:
print("Rows before dropDuplicates:", df.count())
print("Rows after full-row deduplication:", df.dropDuplicates().count())

Rows before dropDuplicates: 120000
Rows after full-row deduplication: 120000


In [75]:
one_per_production = df.dropDuplicates(["production_id"])
print(one_per_production.count())

120000


In [76]:
df.filter(F.col("operator_id").isNotNull()).show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [77]:
df.filter(~(F.col("temperature_c").isNull() & F.col("vibration_mm_s").isNull())).show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [78]:
df.select("product_family").distinct().orderBy("product_family").show()

+--------------+
|product_family|
+--------------+
|   Controllers|
|        Motors|
|         Pumps|
|        Valves|
+--------------+



In [79]:
df.select("plant_id", "machine_id").distinct().orderBy("plant_id", "machine_id").show(50)

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|    PL01|     MC001|
|    PL01|     MC002|
|    PL01|     MC003|
|    PL01|     MC004|
|    PL01|     MC005|
|    PL02|     MC006|
|    PL02|     MC007|
|    PL02|     MC008|
|    PL02|     MC009|
|    PL02|     MC010|
|    PL03|     MC011|
|    PL03|     MC012|
|    PL03|     MC013|
|    PL03|     MC014|
|    PL03|     MC015|
|    PL04|     MC016|
|    PL04|     MC017|
|    PL04|     MC018|
|    PL04|     MC019|
|    PL04|     MC020|
|    PL05|     MC021|
|    PL05|     MC022|
|    PL05|     MC023|
|    PL05|     MC024|
|    PL05|     MC025|
|    PL06|     MC026|
|    PL06|     MC027|
|    PL06|     MC028|
|    PL06|     MC029|
|    PL06|     MC030|
+--------+----------+



7. Sorting


In [80]:
df.orderBy(F.col("units_produced").asc()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

In [81]:
df.orderBy(F.col("units_produced").desc()).show(10)


+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [82]:
df.orderBy(F.col("defective_units").desc()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0051359|     2026-09-16|    PL02|     MC009|     OP0181|Morning|     CTL-230|   Controllers|         1192|          1230|            108|               8|    555.34|        71.97|          2.09|        Rework|              No|     Irfan|
|    PR0108009|     

In [83]:
df.orderBy(F.col("plant_id").asc(), F.col("units_produced").desc()).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|
|    PR0002678|     

In [84]:
df.orderBy(F.col("units_produced").desc()).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [85]:
df.orderBy(F.col("downtime_minutes").desc_nulls_last()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0092818|     2026-02-16|    PL06|     MC027|     OP0016|Morning|     PMP-210|         Pumps|         1066|           907|              2|             281|    359.48|        54.03|           2.8|          Pass|              No|     Farah|
|    PR0045363|     

In [86]:
df.groupBy("machine_id").agg(
    F.max("vibration_mm_s").alias("highest_recorded_vibration")
).orderBy(F.col("highest_recorded_vibration").desc_nulls_last()).show(20)

+----------+--------------------------+
|machine_id|highest_recorded_vibration|
+----------+--------------------------+
|     MC012|                      8.56|
|     MC030|                      8.25|
|     MC027|                      7.85|
|     MC010|                       7.8|
|     MC018|                      7.79|
|     MC024|                      7.77|
|     MC015|                      7.76|
|     MC016|                      7.63|
|     MC007|                      7.61|
|     MC017|                      7.52|
|     MC021|                       7.5|
|     MC004|                      7.48|
|     MC025|                      7.46|
|     MC019|                      7.45|
|     MC028|                      7.43|
|     MC022|                       7.4|
|     MC014|                      7.37|
|     MC006|                      7.36|
|     MC008|                      7.32|
|     MC023|                      7.29|
+----------+--------------------------+
only showing top 20 rows


8. Aggregations

In [87]:
df.agg(
    F.sum("units_planned").alias("total_units_planned"),
    F.sum("units_produced").alias("total_units_produced"),
    F.sum("defective_units").alias("total_defective_units"),
    F.avg("units_produced").alias("average_units_produced"),
    F.max("units_produced").alias("maximum_production_output"),
    F.min("units_produced").alias("minimum_production_output"),
    F.avg("downtime_minutes").alias("average_downtime_minutes"),
    F.sum("energy_kwh").alias("total_energy_consumed"),
    F.avg("temperature_c").alias("average_temperature"),
    F.avg("vibration_mm_s").alias("average_vibration")
).show(truncate=False)

+-------------------+--------------------+---------------------+----------------------+-------------------------+-------------------------+------------------------+---------------------+-------------------+-----------------+
|total_units_planned|total_units_produced|total_defective_units|average_units_produced|maximum_production_output|minimum_production_output|average_downtime_minutes|total_energy_consumed|average_temperature|average_vibration|
+-------------------+--------------------+---------------------+----------------------+-------------------------+-------------------------+------------------------+---------------------+-------------------+-----------------+
|87094621           |80927888            |2030244              |674.3990666666666     |1245                     |205                      |33.93149474428351       |2.8368014080000117E7 |63.021532392167266 |3.398737691604917|
+-------------------+--------------------+---------------------+----------------------+-------------

9. groupBy() summaries

In [88]:
df.groupBy("plant_id").agg(F.sum("units_produced").alias("total_production")).orderBy(F.desc("total_production")).show()

+--------+----------------+
|plant_id|total_production|
+--------+----------------+
|    PL03|        13547769|
|    PL02|        13546452|
|    PL04|        13529935|
|    PL06|        13492337|
|    PL01|        13488232|
|    PL05|        13323163|
+--------+----------------+



In [89]:
df.groupBy("product_family").agg(F.sum("units_produced").alias("total_production")).orderBy(F.desc("total_production")).show()

+--------------+----------------+
|product_family|total_production|
+--------------+----------------+
|        Valves|        20433082|
|         Pumps|        20208739|
|   Controllers|        20168572|
|        Motors|        20117495|
+--------------+----------------+



In [90]:
df.groupBy("plant_id").agg(F.sum("defective_units").alias("total_defects")).orderBy(F.desc("total_defects")).show()

+--------+-------------+
|plant_id|total_defects|
+--------+-------------+
|    PL01|       339720|
|    PL02|       339647|
|    PL03|       339264|
|    PL06|       338993|
|    PL04|       338118|
|    PL05|       334502|
+--------+-------------+



In [91]:
df.groupBy("shift").agg(F.avg("downtime_minutes").alias("average_downtime")).orderBy(F.desc("average_downtime")).show()

+-------+------------------+
|  shift|  average_downtime|
+-------+------------------+
|Evening|  34.2213017151833|
|  Night|33.953058820580935|
|Morning| 33.61973080530278|
+-------+------------------+



In [92]:
df.groupBy("product_family").agg(F.avg("energy_kwh").alias("average_energy_usage")).orderBy(F.desc("average_energy_usage")).show()

+--------------+--------------------+
|product_family|average_energy_usage|
+--------------+--------------------+
|        Motors|  237.21818224597516|
|         Pumps|  236.89137927582252|
|        Valves|  236.34508750495314|
|   Controllers|  235.15370433333288|
+--------------+--------------------+



In [93]:
df.groupBy("quality_status").count().orderBy(F.desc("count")).show()


+--------------+-----+
|quality_status|count|
+--------------+-----+
|          Pass|74328|
|        Rework|22928|
|        Reject|22744|
+--------------+-----+



In [94]:
df.groupBy("plant_id", "shift").agg(F.sum("units_produced").alias("total_production")).orderBy("plant_id", F.desc("total_production")).show(50)

+--------+-------+----------------+
|plant_id|  shift|total_production|
+--------+-------+----------------+
|    PL01|  Night|         4562612|
|    PL01|Evening|         4477769|
|    PL01|Morning|         4447851|
|    PL02|Morning|         4561068|
|    PL02|  Night|         4558467|
|    PL02|Evening|         4426917|
|    PL03|  Night|         4599608|
|    PL03|Evening|         4487211|
|    PL03|Morning|         4460950|
|    PL04|Evening|         4540617|
|    PL04|Morning|         4497099|
|    PL04|  Night|         4492219|
|    PL05|Evening|         4475101|
|    PL05|Morning|         4431446|
|    PL05|  Night|         4416616|
|    PL06|  Night|         4604432|
|    PL06|Evening|         4458798|
|    PL06|Morning|         4429107|
+--------+-------+----------------+



In [95]:
df.groupBy("machine_id").agg(F.avg("temperature_c").alias("average_temperature")).orderBy(F.desc("average_temperature")).show(50)

+----------+-------------------+
|machine_id|average_temperature|
+----------+-------------------+
|     MC019| 63.266223444138944|
|     MC020|  63.22489208633087|
|     MC029|  63.13119731057668|
|     MC009|   63.1252779855396|
|     MC014| 63.114309957574186|
|     MC023| 63.109051746112684|
|     MC027| 63.099668282725126|
|     MC022|  63.09403431245128|
|     MC015|  63.09026488395557|
|     MC001|   63.0778733146782|
|     MC017|  63.07674442190667|
|     MC011| 63.060230414746606|
|     MC007|    63.037661716587|
|     MC016|  63.03639870582378|
|     MC026|  63.02672521957329|
|     MC021|  63.01464069491981|
|     MC005| 62.979236446933044|
|     MC028|  62.97779683034558|
|     MC030|  62.96733774497327|
|     MC024|  62.96457946271644|
|     MC008|  62.96393492628365|
|     MC013| 62.959448777951046|
|     MC004|  62.95817884953549|
|     MC012|  62.95193507800698|
|     MC002|  62.93530761472517|
|     MC006| 62.916547017189075|
|     MC025| 62.898440510283834|
|     MC01

In [96]:
df.groupBy("supervisor").agg(F.sum("units_produced").alias("total_output")).orderBy(F.desc("total_output")).show(50)

+----------+------------+
|supervisor|total_output|
+----------+------------+
|     Farah|    10276986|
|    Harini|    10215803|
|     Deepa|    10130817|
|    Bharat|    10122430|
|     Anita|    10089966|
|    Charan|    10037116|
|     Gopal|    10031531|
|     Irfan|    10023239|
+----------+------------+



10. Joining DataFrames

In [98]:
plant_df = spark.read.csv(
 "plant_master.csv",
 header=True,
 inferSchema=True
)


In [99]:
machine_df = spark.read.csv(
 "machine_master.csv",
 header=True,
 inferSchema=True
)

In [100]:
production_plant = df.join(plant_df, on="plant_id", how="inner")
production_plant.show(10, truncate=False)

+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|plant_id|production_id|production_date|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_name     |city     |region|year_opened|plant_manager|
+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|PL03    |PR0000001    |2026-08-07     |MC011     |OP0275     |Evening|MTR-100     |Moto

In [101]:
production_machine = df.join(machine_df, on=["machine_id", "plant_id"], how="inner")
production_machine.show(10, truncate=False)

+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+------------+-----------------+--------------+
|machine_id|plant_id|production_id|production_date|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|machine_type|capacity_per_hour|installed_year|
+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+------------+-----------------+--------------+
|MC011     |PL03    |PR0000001    |2026-08-07     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           

In [102]:
combined_df = (
    df.join(plant_df, on="plant_id", how="left")
      .join(machine_df, on=["machine_id", "plant_id"], how="left")
)
combined_df.show(10, truncate=False)

+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+------------+-----------------+--------------+
|machine_id|plant_id|production_id|production_date|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_name     |city     |region|year_opened|plant_manager|machine_type|capacity_per_hour|installed_year|
+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-----------

In [104]:
combined_df.groupBy("city").agg(F.sum("units_produced").alias("total_production")).orderBy(F.desc("total_production")).show()

+---------+----------------+
|     city|total_production|
+---------+----------------+
|  Chennai|        13547769|
|     Pune|        13546452|
|    Noida|        13529935|
|  Kolkata|        13492337|
|Hyderabad|        13488232|
|Ahmedabad|        13323163|
+---------+----------------+



In [105]:
combined_df.groupBy("machine_type").agg(F.avg("downtime_minutes").alias("average_downtime")).orderBy(F.desc("average_downtime")).show()

+------------+-----------------+
|machine_type| average_downtime|
+------------+-----------------+
|     Casting| 34.1660783396419|
|     Welding|34.00449382099613|
|    Assembly|33.96718284443325|
|     Testing|33.87585594724829|
|         CNC|33.64548129466162|
+------------+-----------------+



In [106]:
combined_df.groupBy("region").agg(F.sum("defective_units").alias("total_defects")).orderBy(F.desc("total_defects")).show()

+------+-------------+
|region|total_defects|
+------+-------------+
| South|       678984|
|  West|       674149|
|  East|       338993|
| North|       338118|
+------+-------------+



In [107]:
unmatched_machine = df.join(
    machine_df.select("machine_id", "plant_id").dropDuplicates(),
    on=["machine_id", "plant_id"],
    how="left_anti"
)
unmatched_machine.count()
unmatched_machine.show(20, truncate=False)

+----------+--------+-------------+---------------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|machine_id|plant_id|production_id|production_date|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+----------+--------+-------------+---------------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
+----------+--------+-------------+---------------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+



In [108]:
left_machine_join = df.join(
    machine_df.withColumnRenamed("machine_type", "master_machine_type"),
    on=["machine_id", "plant_id"],
    how="left"
)
left_machine_join.filter(F.col("master_machine_type").isNull()).show(20, truncate=False)

+----------+--------+-------------+---------------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-------------------+-----------------+--------------+
|machine_id|plant_id|production_id|production_date|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|master_machine_type|capacity_per_hour|installed_year|
+----------+--------+-------------+---------------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-------------------+-----------------+--------------+
+----------+--------+-------------+---------------+-----------+-----+------------+--------------+-------------+-

In [109]:
combined_df.groupBy("plant_name").agg(F.sum("energy_kwh").alias("total_energy_consumption")).orderBy(F.desc("total_energy_consumption")).show()

+---------------+------------------------+
|     plant_name|total_energy_consumption|
+---------------+------------------------+
|     Pune Plant|       4757984.949999988|
|    Noida Plant|       4746238.580000002|
|  Chennai Plant|       4742748.169999995|
|Hyderabad Plant|       4725471.189999996|
|  Kolkata Plant|       4723180.160000009|
|Ahmedabad Plant|        4672391.03000001|
+---------------+------------------------+



11. Updating Data

In [110]:
df = df.withColumn(
    "quality_status",
    F.when(F.col("quality_status") == "Reject", "Failed")
     .otherwise(F.col("quality_status"))
)

df.select("production_id", "quality_status").show(10)

+-------------+--------------+
|production_id|quality_status|
+-------------+--------------+
|    PR0000001|        Rework|
|    PR0000002|        Failed|
|    PR0000003|        Failed|
|    PR0000004|        Rework|
|    PR0000005|          Pass|
|    PR0000006|        Rework|
|    PR0000007|          Pass|
|    PR0000008|        Failed|
|    PR0000009|          Pass|
|    PR0000010|        Failed|
+-------------+--------------+
only showing top 10 rows


In [111]:
df = df.withColumn(
    "quality_status",
    F.when(F.col("quality_status") == "Rework", "Requires Review")
     .otherwise(F.col("quality_status"))
)

df.select("production_id", "quality_status").show(10)

+-------------+---------------+
|production_id| quality_status|
+-------------+---------------+
|    PR0000001|Requires Review|
|    PR0000002|         Failed|
|    PR0000003|         Failed|
|    PR0000004|Requires Review|
|    PR0000005|           Pass|
|    PR0000006|Requires Review|
|    PR0000007|           Pass|
|    PR0000008|         Failed|
|    PR0000009|           Pass|
|    PR0000010|         Failed|
+-------------+---------------+
only showing top 10 rows


In [112]:
df = df.withColumn(
    "units_planned",
    F.when(
        F.col("product_family") == "Motors",
        F.col("units_planned") * 1.05
    ).otherwise(F.col("units_planned"))
)

df.select("product_family", "units_planned").show(10)

+--------------+------------------+
|product_family|     units_planned|
+--------------+------------------+
|        Motors|            1113.0|
|   Controllers|             809.0|
|        Motors|             464.1|
|         Pumps|             787.0|
|         Pumps|             344.0|
|        Motors|1005.9000000000001|
|        Valves|            1165.0|
|        Valves|             862.0|
|   Controllers|             313.0|
|   Controllers|             532.0|
+--------------+------------------+
only showing top 10 rows


In [113]:
df = df.withColumn(
    "energy_kwh",
    F.when(
        F.col("shift") == "Night",
        F.col("energy_kwh") * 1.03
    ).otherwise(F.col("energy_kwh"))
)

df.select("shift", "energy_kwh").show(10)

+-------+----------+
|  shift|energy_kwh|
+-------+----------+
|Evening|    388.11|
|Morning|    398.58|
|Evening|    183.34|
|Morning|    217.61|
|Morning|    115.47|
|Evening|    290.26|
|Morning|    355.56|
|  Night|  274.0418|
|Evening|     87.62|
|Evening|    112.17|
+-------+----------+
only showing top 10 rows


In [114]:
df = df.withColumn(
    "operator_id",
    F.coalesce(F.col("operator_id"), F.lit("UNASSIGNED"))
)

df.select("production_id", "operator_id").show(10)

+-------------+-----------+
|production_id|operator_id|
+-------------+-----------+
|    PR0000001|     OP0275|
|    PR0000002|     OP0489|
|    PR0000003|     OP0189|
|    PR0000004|     OP0076|
|    PR0000005|     OP0501|
|    PR0000006|     OP0501|
|    PR0000007|     OP0645|
|    PR0000008|     OP0106|
|    PR0000009|     OP0386|
|    PR0000010|     OP0593|
+-------------+-----------+
only showing top 10 rows


In [115]:
df = df.withColumn(
    "supervisor",
    F.when(
        F.col("supervisor") == "Anita",
        "Anita Rao"
    ).otherwise(F.col("supervisor"))
)

df.select("production_id", "supervisor").show(10)

+-------------+----------+
|production_id|supervisor|
+-------------+----------+
|    PR0000001|    Charan|
|    PR0000002|     Farah|
|    PR0000003|     Deepa|
|    PR0000004|     Farah|
|    PR0000005|     Deepa|
|    PR0000006|     Gopal|
|    PR0000007|     Gopal|
|    PR0000008|     Irfan|
|    PR0000009| Anita Rao|
|    PR0000010|     Gopal|
+-------------+----------+
only showing top 10 rows


In [116]:
df = df.withColumn(
    "corrected_temperature_c",
    F.coalesce(F.col("temperature_c"), F.lit(0.0))
)

df.select(
    "production_id",
    "temperature_c",
    "corrected_temperature_c"
).show(10)

+-------------+-------------+-----------------------+
|production_id|temperature_c|corrected_temperature_c|
+-------------+-------------+-----------------------+
|    PR0000001|         61.4|                   61.4|
|    PR0000002|        64.37|                  64.37|
|    PR0000003|        54.13|                  54.13|
|    PR0000004|        64.47|                  64.47|
|    PR0000005|        72.59|                  72.59|
|    PR0000006|        64.91|                  64.91|
|    PR0000007|        62.76|                  62.76|
|    PR0000008|         57.1|                   57.1|
|    PR0000009|        66.55|                  66.55|
|    PR0000010|        60.96|                  60.96|
+-------------+-------------+-----------------------+
only showing top 10 rows


12. Null values and na functions

In [118]:
df.filter(F.col("operator_id").isNull()).count()

0

In [119]:
df.filter(F.col("temperature_c").isNull()).count()

2132

In [120]:
df.filter(F.col("vibration_mm_s").isNull()).count()

1788

In [121]:
df.filter(F.col("downtime_minutes").isNull()).count()

1177

In [122]:
df.na.drop().count()

114985

In [123]:
df.na.drop(subset=["temperature_c"]).count()

117868

In [124]:
downtime_filled = df.na.fill({"downtime_minutes": 0})
operator_filled = df.na.fill({"operator_id": "UNKNOWN"})

In [125]:
average_temperature = df.select(F.avg("temperature_c").alias("avg_temp")).first()["avg_temp"]
temperature_filled = df.na.fill({"temperature_c": float(average_temperature)})

In [126]:
print(average_temperature)
print(df.count())

63.021532392167266
120000


In [127]:
cleansed_df = (
    df.na.fill({"downtime_minutes": 0, "operator_id": "UNKNOWN"})
      .na.fill({"temperature_c": float(average_temperature)})
)

In [128]:
print(cleansed_df.count())

120000


In [129]:
df.select(
    F.count(F.when(F.col("operator_id").isNull(), 1)).alias("missing_operator_ids"),
    F.count(F.when(F.col("temperature_c").isNull(), 1)).alias("missing_temperatures"),
    F.count(F.when(F.col("vibration_mm_s").isNull(), 1)).alias("missing_vibrations"),
    F.count(F.when(F.col("downtime_minutes").isNull(), 1)).alias("missing_downtime")
).show()


+--------------------+--------------------+------------------+----------------+
|missing_operator_ids|missing_temperatures|missing_vibrations|missing_downtime|
+--------------------+--------------------+------------------+----------------+
|                   0|                2132|              1788|            1177|
+--------------------+--------------------+------------------+----------------+



13. Built-in Functions

In [131]:
df.select(F.upper("product_family").alias("product_family_upper")).show(10)

+--------------------+
|product_family_upper|
+--------------------+
|              MOTORS|
|         CONTROLLERS|
|              MOTORS|
|               PUMPS|
|               PUMPS|
|              MOTORS|
|              VALVES|
|              VALVES|
|         CONTROLLERS|
|         CONTROLLERS|
+--------------------+
only showing top 10 rows


In [132]:
df.select(F.lower("supervisor").alias("supervisor_lower")).show(10)

+----------------+
|supervisor_lower|
+----------------+
|          charan|
|           farah|
|           deepa|
|           farah|
|           deepa|
|           gopal|
|           gopal|
|           irfan|
|       anita rao|
|           gopal|
+----------------+
only showing top 10 rows


In [133]:
df.select(F.trim("operator_id").alias("operator_id_trimmed")).show(10)

+-------------------+
|operator_id_trimmed|
+-------------------+
|             OP0275|
|             OP0489|
|             OP0189|
|             OP0076|
|             OP0501|
|             OP0501|
|             OP0645|
|             OP0106|
|             OP0386|
|             OP0593|
+-------------------+
only showing top 10 rows


In [134]:
df.select("product_code", F.length("product_code").alias("product_code_length")).show(10)

+------------+-------------------+
|product_code|product_code_length|
+------------+-------------------+
|     MTR-100|                  7|
|     CTL-330|                  7|
|     MTR-300|                  7|
|     PMP-210|                  7|
|     PMP-110|                  7|
|     MTR-300|                  7|
|     VLV-120|                  7|
|     VLV-320|                  7|
|     CTL-130|                  7|
|     CTL-230|                  7|
+------------+-------------------+
only showing top 10 rows


In [135]:
df.select(
    "plant_id", "machine_id",
    F.concat(F.col("plant_id"), F.lit("_"), F.col("machine_id")).alias("plant_machine")
).show(10)

+--------+----------+-------------+
|plant_id|machine_id|plant_machine|
+--------+----------+-------------+
|    PL03|     MC011|   PL03_MC011|
|    PL04|     MC019|   PL04_MC019|
|    PL01|     MC002|   PL01_MC002|
|    PL03|     MC013|   PL03_MC013|
|    PL04|     MC020|   PL04_MC020|
|    PL01|     MC002|   PL01_MC002|
|    PL02|     MC007|   PL02_MC007|
|    PL02|     MC009|   PL02_MC009|
|    PL03|     MC015|   PL03_MC015|
|    PL02|     MC006|   PL02_MC006|
+--------+----------+-------------+
only showing top 10 rows


In [136]:
df.select(F.round("temperature_c", 1).alias("temperature_rounded")).show(10)

+-------------------+
|temperature_rounded|
+-------------------+
|               61.4|
|               64.4|
|               54.1|
|               64.5|
|               72.6|
|               64.9|
|               62.8|
|               57.1|
|               66.6|
|               61.0|
+-------------------+
only showing top 10 rows


In [137]:
df.select(
    "production_id",
    F.abs(F.col("units_planned") - F.col("units_produced")).alias("planned_produced_difference")
).show(10)

+-------------+---------------------------+
|production_id|planned_produced_difference|
+-------------+---------------------------+
|    PR0000001|                      120.0|
|    PR0000002|                       27.0|
|    PR0000003|          84.10000000000002|
|    PR0000004|                       67.0|
|    PR0000005|                       48.0|
|    PR0000006|          210.9000000000001|
|    PR0000007|                      210.0|
|    PR0000008|                       33.0|
|    PR0000009|                       24.0|
|    PR0000010|                        5.0|
+-------------+---------------------------+
only showing top 10 rows


In [138]:
df.select(
    "production_id",
    F.regexp_replace("production_id", "PR", "PROD-").alias("production_id_replaced")
).show(10)

+-------------+----------------------+
|production_id|production_id_replaced|
+-------------+----------------------+
|    PR0000001|          PROD-0000001|
|    PR0000002|          PROD-0000002|
|    PR0000003|          PROD-0000003|
|    PR0000004|          PROD-0000004|
|    PR0000005|          PROD-0000005|
|    PR0000006|          PROD-0000006|
|    PR0000007|          PROD-0000007|
|    PR0000008|          PROD-0000008|
|    PR0000009|          PROD-0000009|
|    PR0000010|          PROD-0000010|
+-------------+----------------------+
only showing top 10 rows


In [139]:
df.select(
    "product_code",
    F.substring("product_code", 1, 3).alias("product_code_prefix")
).show(10)

+------------+-------------------+
|product_code|product_code_prefix|
+------------+-------------------+
|     MTR-100|                MTR|
|     CTL-330|                CTL|
|     MTR-300|                MTR|
|     PMP-210|                PMP|
|     PMP-110|                PMP|
|     MTR-300|                MTR|
|     VLV-120|                VLV|
|     VLV-320|                VLV|
|     CTL-130|                CTL|
|     CTL-230|                CTL|
+------------+-------------------+
only showing top 10 rows


In [140]:
df.select(
    F.coalesce(F.col("temperature_c"), F.lit(25.0)).alias("temperature_default_25")
).show(10)

+----------------------+
|temperature_default_25|
+----------------------+
|                  61.4|
|                 64.37|
|                 54.13|
|                 64.47|
|                 72.59|
|                 64.91|
|                 62.76|
|                  57.1|
|                 66.55|
|                 60.96|
+----------------------+
only showing top 10 rows


15. Complex JSON


In [141]:
machine_json = spark.read.option(
 "multiline", "true"
).json("machine_configurations.json")


In [142]:
machine_json.select("machine_id", "machine_type").show(50, truncate=False)

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|MC001     |CNC         |
|MC002     |Assembly    |
|MC003     |Casting     |
|MC004     |Welding     |
|MC005     |Testing     |
|MC006     |CNC         |
|MC007     |Assembly    |
|MC008     |Casting     |
|MC009     |Welding     |
|MC010     |Testing     |
|MC011     |CNC         |
|MC012     |Assembly    |
|MC013     |Casting     |
|MC014     |Welding     |
|MC015     |Testing     |
|MC016     |CNC         |
|MC017     |Assembly    |
|MC018     |Casting     |
|MC019     |Welding     |
|MC020     |Testing     |
|MC021     |CNC         |
|MC022     |Assembly    |
|MC023     |Casting     |
|MC024     |Welding     |
|MC025     |Testing     |
|MC026     |CNC         |
|MC027     |Assembly    |
|MC028     |Casting     |
|MC029     |Welding     |
|MC030     |Testing     |
+----------+------------+



In [144]:
machine_json.select(
    "machine_id",
    F.col("specifications.max_temperature_c").alias("maximum_temperature_c")
).show(50)

+----------+---------------------+
|machine_id|maximum_temperature_c|
+----------+---------------------+
|     MC001|                   85|
|     MC002|                   85|
|     MC003|                   90|
|     MC004|                   85|
|     MC005|                   85|
|     MC006|                   85|
|     MC007|                   80|
|     MC008|                   75|
|     MC009|                   85|
|     MC010|                   90|
|     MC011|                   90|
|     MC012|                   80|
|     MC013|                   90|
|     MC014|                   80|
|     MC015|                   90|
|     MC016|                   75|
|     MC017|                   85|
|     MC018|                   85|
|     MC019|                   75|
|     MC020|                   80|
|     MC021|                   80|
|     MC022|                   75|
|     MC023|                   90|
|     MC024|                   90|
|     MC025|                   80|
|     MC026|        

In [145]:
machine_json.select(
    "machine_id",
    F.col("specifications.vibration_limit_mm_s").alias("vibration_limit_mm_s")
).show(50)

+----------+--------------------+
|machine_id|vibration_limit_mm_s|
+----------+--------------------+
|     MC001|                 5.0|
|     MC002|                 5.5|
|     MC003|                 4.5|
|     MC004|                 4.5|
|     MC005|                 4.5|
|     MC006|                 5.5|
|     MC007|                 4.5|
|     MC008|                 5.0|
|     MC009|                 5.5|
|     MC010|                 4.5|
|     MC011|                 6.0|
|     MC012|                 5.5|
|     MC013|                 5.0|
|     MC014|                 5.5|
|     MC015|                 4.5|
|     MC016|                 5.5|
|     MC017|                 4.5|
|     MC018|                 5.0|
|     MC019|                 5.0|
|     MC020|                 5.0|
|     MC021|                 4.5|
|     MC022|                 5.5|
|     MC023|                 5.5|
|     MC024|                 5.5|
|     MC025|                 4.5|
|     MC026|                 4.5|
|     MC027|  

In [146]:
machine_json.select(
    "machine_id",
    F.col("maintenance.frequency_days").alias("maintenance_frequency_days")
).show(50)

+----------+--------------------------+
|machine_id|maintenance_frequency_days|
+----------+--------------------------+
|     MC001|                        60|
|     MC002|                        60|
|     MC003|                        45|
|     MC004|                        60|
|     MC005|                        30|
|     MC006|                        60|
|     MC007|                        30|
|     MC008|                        30|
|     MC009|                        30|
|     MC010|                        45|
|     MC011|                        30|
|     MC012|                        30|
|     MC013|                        60|
|     MC014|                        90|
|     MC015|                        30|
|     MC016|                        45|
|     MC017|                        30|
|     MC018|                        30|
|     MC019|                        30|
|     MC020|                        45|
|     MC021|                        60|
|     MC022|                        90|


In [147]:
machine_json.select("machine_id", "sensors").show(5, truncate=False)
sensor_rows = machine_json.select(
    "machine_id", "plant_id", F.explode("sensors").alias("sensor_info")
)

+----------+----------------------------------------------------------------------+
|machine_id|sensors                                                               |
+----------+----------------------------------------------------------------------+
|MC001     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC002     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC003     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC004     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC005     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
+----------+----------------------------------------------------------------------+
only showing top 5 rows


In [148]:
sensor_rows.select(
    "machine_id",
    "plant_id",
    F.col("sensor_info.sensor").alias("sensor"),
    F.col("sensor_info.unit").alias("unit"),
    F.col("sensor_info.active").alias("active")
).show(100, truncate=False)


+----------+--------+-----------+----+------+
|machine_id|plant_id|sensor     |unit|active|
+----------+--------+-----------+----+------+
|MC001     |PL01    |temperature|C   |true  |
|MC001     |PL01    |vibration  |mm/s|true  |
|MC001     |PL01    |energy     |kWh |true  |
|MC002     |PL01    |temperature|C   |true  |
|MC002     |PL01    |vibration  |mm/s|true  |
|MC002     |PL01    |energy     |kWh |true  |
|MC003     |PL01    |temperature|C   |true  |
|MC003     |PL01    |vibration  |mm/s|true  |
|MC003     |PL01    |energy     |kWh |true  |
|MC004     |PL01    |temperature|C   |true  |
|MC004     |PL01    |vibration  |mm/s|true  |
|MC004     |PL01    |energy     |kWh |true  |
|MC005     |PL01    |temperature|C   |true  |
|MC005     |PL01    |vibration  |mm/s|true  |
|MC005     |PL01    |energy     |kWh |true  |
|MC006     |PL02    |temperature|C   |true  |
|MC006     |PL02    |vibration  |mm/s|true  |
|MC006     |PL02    |energy     |kWh |true  |
|MC007     |PL02    |temperature|C

In [149]:
active_temperature_sensors = sensor_rows.filter(
    (F.col("sensor_info.sensor") == "temperature") &
    (F.col("sensor_info.active") == True)
)
active_temperature_sensors.select("machine_id", "plant_id", "sensor_info").show(50, truncate=False)

+----------+--------+----------------------+
|machine_id|plant_id|sensor_info           |
+----------+--------+----------------------+
|MC001     |PL01    |{true, temperature, C}|
|MC002     |PL01    |{true, temperature, C}|
|MC003     |PL01    |{true, temperature, C}|
|MC004     |PL01    |{true, temperature, C}|
|MC005     |PL01    |{true, temperature, C}|
|MC006     |PL02    |{true, temperature, C}|
|MC007     |PL02    |{true, temperature, C}|
|MC008     |PL02    |{true, temperature, C}|
|MC009     |PL02    |{true, temperature, C}|
|MC010     |PL02    |{true, temperature, C}|
|MC011     |PL03    |{true, temperature, C}|
|MC012     |PL03    |{true, temperature, C}|
|MC013     |PL03    |{true, temperature, C}|
|MC014     |PL03    |{true, temperature, C}|
|MC015     |PL03    |{true, temperature, C}|
|MC016     |PL04    |{true, temperature, C}|
|MC017     |PL04    |{true, temperature, C}|
|MC018     |PL04    |{true, temperature, C}|
|MC019     |PL04    |{true, temperature, C}|
|MC020    

In [150]:
production_with_config = df.join(
    machine_json.select(
        "machine_id", "plant_id", "machine_type",
        F.col("specifications.rated_power_kw").alias("rated_power_kw"),
        F.col("specifications.max_temperature_c").alias("max_temperature_c"),
        F.col("specifications.vibration_limit_mm_s").alias("vibration_limit_mm_s"),
        F.col("maintenance.frequency_days").alias("maintenance_frequency_days")
    ),
    on=["machine_id", "plant_id"],
    how="left"
)
production_with_config.show(10, truncate=False)

+----------+--------+-------------+---------------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+-----------------------+------------+--------------+-----------------+--------------------+--------------------------+
|machine_id|plant_id|production_id|production_date|operator_id|shift  |product_code|product_family|units_planned     |units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status |maintenance_flag|supervisor|corrected_temperature_c|machine_type|rated_power_kw|max_temperature_c|vibration_limit_mm_s|maintenance_frequency_days|
+----------+--------+-------------+---------------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+------------

16. Union and Union ALL

In [152]:
extra_df = spark.read.csv(
 "manufacturing_production_extra_12k.csv",
 header=True,
 inferSchema=True
)


In [153]:
main_count = df.count()
extra_count = extra_df.count()
print("Main count:", main_count)
print("Extra count:", extra_count)

Main count: 120000
Extra count: 12500


In [155]:
extra_df = extra_df.withColumn(
    "corrected_temperature_c",
    F.coalesce(F.col("temperature_c"), F.lit(0.0))
)

In [156]:
union_df = df.union(extra_df)
union_all_df = df.unionAll(extra_df)
print("union() count:", union_df.count())
print("unionAll() count:", union_all_df.count())


union() count: 132500
unionAll() count: 132500


In [157]:
duplicate_ids = (
    union_df.groupBy("production_id")
    .count()
    .filter(F.col("count") > 1)
)
duplicate_ids.show(50, truncate=False)
duplicate_ids.count()

+-------------+-----+
|production_id|count|
+-------------+-----+
|PR0000413    |2    |
|PR0000135    |2    |
|PR0000236    |2    |
|PR0000339    |2    |
|PR0000059    |2    |
|PR0000463    |2    |
|PR0000136    |2    |
|PR0000263    |2    |
|PR0000393    |2    |
|PR0000445    |2    |
|PR0000124    |2    |
|PR0000299    |2    |
|PR0000341    |2    |
|PR0000008    |2    |
|PR0000172    |2    |
|PR0000359    |2    |
|PR0000389    |2    |
|PR0000026    |2    |
|PR0000274    |2    |
|PR0000161    |2    |
|PR0000031    |2    |
|PR0000086    |2    |
|PR0000120    |2    |
|PR0000128    |2    |
|PR0000402    |2    |
|PR0000013    |2    |
|PR0000067    |2    |
|PR0000383    |2    |
|PR0000418    |2    |
|PR0000068    |2    |
|PR0000200    |2    |
|PR0000245    |2    |
|PR0000077    |2    |
|PR0000260    |2    |
|PR0000472    |2    |
|PR0000491    |2    |
|PR0000095    |2    |
|PR0000184    |2    |
|PR0000199    |2    |
|PR0000293    |2    |
|PR0000400    |2    |
|PR0000002    |2    |
|PR0000197

500

In [158]:
deduplicated_union = union_df.dropDuplicates(["production_id"])
deduplicated_union.count()

132000

In [159]:
union_by_name_df = df.unionByName(extra_df)
union_by_name_df.count()

132500

In [160]:
reordered_extra = extra_df.select(*list(reversed(extra_df.columns)))
union_by_name_reordered = df.unionByName(reordered_extra)
union_by_name_reordered.count()

132500

17. Windows Function

In [162]:
from pyspark.sql.window import Window
plant_output_window = Window.partitionBy("plant_id").orderBy(F.col("units_produced").desc())
ranked_df = (
    df.withColumn("row_number", F.row_number().over(plant_output_window))
      .withColumn("rank", F.rank().over(plant_output_window))
      .withColumn("dense_rank", F.dense_rank().over(plant_output_window))
)

In [163]:
ranked_df.select(
    "production_id", "plant_id", "machine_id", "units_produced",
    "row_number", "rank", "dense_rank"
).orderBy("plant_id", "row_number").show(50, truncate=False)


+-------------+--------+----------+--------------+----------+----+----------+
|production_id|plant_id|machine_id|units_produced|row_number|rank|dense_rank|
+-------------+--------+----------+--------------+----------+----+----------+
|PR0073059    |PL01    |MC003     |1239          |1         |1   |1         |
|PR0002678    |PL01    |MC005     |1238          |2         |2   |2         |
|PR0005192    |PL01    |MC002     |1236          |3         |3   |3         |
|PR0044516    |PL01    |MC002     |1236          |4         |3   |3         |
|PR0056319    |PL01    |MC002     |1236          |5         |3   |3         |
|PR0006350    |PL01    |MC004     |1234          |6         |6   |4         |
|PR0046906    |PL01    |MC005     |1233          |7         |7   |5         |
|PR0046582    |PL01    |MC002     |1230          |8         |8   |6         |
|PR0024941    |PL01    |MC002     |1229          |9         |9   |7         |
|PR0088933    |PL01    |MC003     |1229          |10        |9  

In [164]:
ranked_df.filter(F.col("row_number") <= 5).orderBy("plant_id", "row_number").show(100, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+-----------------+-------------+--------------+---------------+----------------+----------+-----------------------+----------+----+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned     |units_produced|defective_units|downtime_minutes|energy_kwh       |temperature_c|vibration_mm_s|quality_status |maintenance_flag|supervisor|corrected_temperature_c|row_number|rank|dense_rank|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+-----------------+-------------+--------------+---------------+----------------+----------+-----------------------+----------+----+----------+
|PR0073059    |2026-04-01     |PL01    |MC003     |OP0110     |Evening|MT

In [165]:
machine_totals = df.groupBy("plant_id", "machine_id").agg(
    F.sum("units_produced").alias("machine_total_output")
)

In [166]:
machine_rank_window = Window.partitionBy("plant_id").orderBy(F.col("machine_total_output").desc())
machine_ranked = machine_totals.withColumn("machine_rank", F.dense_rank().over(machine_rank_window))
machine_ranked.filter(F.col("machine_rank") <= 3).orderBy("plant_id", "machine_rank").show(100, truncate=False)

+--------+----------+--------------------+------------+
|plant_id|machine_id|machine_total_output|machine_rank|
+--------+----------+--------------------+------------+
|PL01    |MC004     |2722412             |1           |
|PL01    |MC002     |2713779             |2           |
|PL01    |MC001     |2691431             |3           |
|PL02    |MC009     |2760070             |1           |
|PL02    |MC007     |2713038             |2           |
|PL02    |MC008     |2705850             |3           |
|PL03    |MC012     |2750570             |1           |
|PL03    |MC015     |2739284             |2           |
|PL03    |MC014     |2735500             |3           |
|PL04    |MC016     |2750780             |1           |
|PL04    |MC019     |2735839             |2           |
|PL04    |MC017     |2726300             |3           |
|PL05    |MC024     |2737331             |1           |
|PL05    |MC023     |2718075             |2           |
|PL05    |MC022     |2635629             |3     

In [167]:
plant_total_window = Window.partitionBy("plant_id")
df.withColumn("plant_total_production", F.sum("units_produced").over(plant_total_window)).select(
    "production_id", "plant_id", "units_produced", "plant_total_production"
).show(10, truncate=False)

+-------------+--------+--------------+----------------------+
|production_id|plant_id|units_produced|plant_total_production|
+-------------+--------+--------------+----------------------+
|PR0000003    |PL01    |380           |13488232              |
|PR0000006    |PL01    |795           |13488232              |
|PR0000019    |PL01    |380           |13488232              |
|PR0000026    |PL01    |663           |13488232              |
|PR0000028    |PL01    |1042          |13488232              |
|PR0000030    |PL01    |359           |13488232              |
|PR0000031    |PL01    |776           |13488232              |
|PR0000039    |PL01    |265           |13488232              |
|PR0000040    |PL01    |306           |13488232              |
|PR0000050    |PL01    |908           |13488232              |
+-------------+--------+--------------+----------------------+
only showing top 10 rows


In [168]:
machine_window = Window.partitionBy("machine_id")
df.withColumn("average_machine_output", F.avg("units_produced").over(machine_window)).select(
    "production_id", "machine_id", "units_produced", "average_machine_output"
).show(10, truncate=False)

+-------------+----------+--------------+----------------------+
|production_id|machine_id|units_produced|average_machine_output|
+-------------+----------+--------------+----------------------+
|PR0000026    |MC001     |663           |671.5147205588822     |
|PR0000057    |MC001     |325           |671.5147205588822     |
|PR0000088    |MC001     |676           |671.5147205588822     |
|PR0000135    |MC001     |375           |671.5147205588822     |
|PR0000139    |MC001     |378           |671.5147205588822     |
|PR0000159    |MC001     |762           |671.5147205588822     |
|PR0000216    |MC001     |431           |671.5147205588822     |
|PR0000232    |MC001     |1030          |671.5147205588822     |
|PR0000243    |MC001     |464           |671.5147205588822     |
|PR0000279    |MC001     |616           |671.5147205588822     |
+-------------+----------+--------------+----------------------+
only showing top 10 rows


In [169]:
machine_date_window = Window.partitionBy("machine_id").orderBy("production_date", "production_id")
date_ordered_df = df.withColumn("production_date", F.to_date("production_date"))

In [170]:
date_ordered_df.withColumn(
    "cumulative_machine_production",
    F.sum("units_produced").over(
        machine_date_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
).select("production_id", "machine_id", "production_date", "units_produced", "cumulative_machine_production").show(20, truncate=False)

+-------------+----------+---------------+--------------+-----------------------------+
|production_id|machine_id|production_date|units_produced|cumulative_machine_production|
+-------------+----------+---------------+--------------+-----------------------------+
|PR0002356    |MC001     |2026-01-01     |437           |437                          |
|PR0030520    |MC001     |2026-01-01     |997           |1434                         |
|PR0042532    |MC001     |2026-01-01     |822           |2256                         |
|PR0050348    |MC001     |2026-01-01     |817           |3073                         |
|PR0057063    |MC001     |2026-01-01     |784           |3857                         |
|PR0069086    |MC001     |2026-01-01     |472           |4329                         |
|PR0072621    |MC001     |2026-01-01     |1078          |5407                         |
|PR0082917    |MC001     |2026-01-01     |1027          |6434                         |
|PR0088125    |MC001     |2026-0

In [171]:
with_previous_next = (
    date_ordered_df
    .withColumn("previous_output", F.lag("units_produced").over(machine_date_window))
    .withColumn("next_output", F.lead("units_produced").over(machine_date_window))
    .withColumn("change_from_previous", F.col("units_produced") - F.col("previous_output"))
)
with_previous_next.select(
    "production_id", "machine_id", "production_date", "units_produced",
    "previous_output", "next_output", "change_from_previous"
).show(20, truncate=False)

+-------------+----------+---------------+--------------+---------------+-----------+--------------------+
|production_id|machine_id|production_date|units_produced|previous_output|next_output|change_from_previous|
+-------------+----------+---------------+--------------+---------------+-----------+--------------------+
|PR0002356    |MC001     |2026-01-01     |437           |NULL           |997        |NULL                |
|PR0030520    |MC001     |2026-01-01     |997           |437            |822        |560                 |
|PR0042532    |MC001     |2026-01-01     |822           |997            |817        |-175                |
|PR0050348    |MC001     |2026-01-01     |817           |822            |784        |-5                  |
|PR0057063    |MC001     |2026-01-01     |784           |817            |472        |-33                 |
|PR0069086    |MC001     |2026-01-01     |472           |784            |1078       |-312                |
|PR0072621    |MC001     |2026-01-01 

In [172]:
family_totals = df.groupBy("product_family").agg(F.sum("units_produced").alias("total_output"))
family_rank_window = Window.orderBy(F.col("total_output").desc())
family_totals.withColumn("family_rank", F.dense_rank().over(family_rank_window)).show()

+--------------+------------+-----------+
|product_family|total_output|family_rank|
+--------------+------------+-----------+
|        Valves|    20433082|          1|
|         Pumps|    20208739|          2|
|   Controllers|    20168572|          3|
|        Motors|    20117495|          4|
+--------------+------------+-----------+



18. Date and Time Fucntions

In [173]:
dated_df = df.withColumn("production_date", F.to_date("production_date"))

In [174]:
date_parts_df = (
    dated_df
    .withColumn("year", F.year("production_date"))
    .withColumn("month", F.month("production_date"))
    .withColumn("day", F.dayofmonth("production_date"))
    .withColumn("month_name", F.date_format("production_date", "MMMM"))
    .withColumn("day_name", F.date_format("production_date", "EEEE"))
    .withColumn("year_month", F.date_format("production_date", "yyyy-MM"))
)

In [175]:
date_parts_df.select(
    "production_id", "production_date", "year", "month", "day",
    "month_name", "day_name", "year_month"
).show(20, truncate=False)


+-------------+---------------+----+-----+---+----------+---------+----------+
|production_id|production_date|year|month|day|month_name|day_name |year_month|
+-------------+---------------+----+-----+---+----------+---------+----------+
|PR0000001    |2026-08-07     |2026|8    |7  |August    |Friday   |2026-08   |
|PR0000002    |2026-08-03     |2026|8    |3  |August    |Monday   |2026-08   |
|PR0000003    |2026-05-20     |2026|5    |20 |May       |Wednesday|2026-05   |
|PR0000004    |2026-01-15     |2026|1    |15 |January   |Thursday |2026-01   |
|PR0000005    |2026-09-21     |2026|9    |21 |September |Monday   |2026-09   |
|PR0000006    |2026-09-03     |2026|9    |3  |September |Thursday |2026-09   |
|PR0000007    |2026-05-07     |2026|5    |7  |May       |Thursday |2026-05   |
|PR0000008    |2026-08-17     |2026|8    |17 |August    |Monday   |2026-08   |
|PR0000009    |2026-03-10     |2026|3    |10 |March     |Tuesday  |2026-03   |
|PR0000010    |2026-02-13     |2026|2    |13 |Februa

In [176]:
date_parts_df.groupBy("year_month").agg(
    F.sum("units_produced").alias("monthly_production")
).orderBy("year_month").show(100)

+----------+------------------+
|year_month|monthly_production|
+----------+------------------+
|   2026-01|           8478059|
|   2026-02|           7810891|
|   2026-03|           8441121|
|   2026-04|           8211005|
|   2026-05|           8507542|
|   2026-06|           8400505|
|   2026-07|           8476416|
|   2026-08|           8529150|
|   2026-09|           8115221|
|   2026-10|           5957978|
+----------+------------------+



In [177]:
date_parts_df.groupBy("year_month").agg(
    F.sum("defective_units").alias("monthly_defects")
).orderBy("year_month").show(100)

+----------+---------------+
|year_month|monthly_defects|
+----------+---------------+
|   2026-01|         211741|
|   2026-02|         197462|
|   2026-03|         212036|
|   2026-04|         208707|
|   2026-05|         212049|
|   2026-06|         210381|
|   2026-07|         210482|
|   2026-08|         214807|
|   2026-09|         202990|
|   2026-10|         149589|
+----------+---------------+



In [178]:
date_parts_df.groupBy("year_month", "plant_id").agg(
    F.sum("units_produced").alias("monthly_production")
).orderBy("year_month", F.desc("monthly_production")).show(100)

+----------+--------+------------------+
|year_month|plant_id|monthly_production|
+----------+--------+------------------+
|   2026-01|    PL01|           1451436|
|   2026-01|    PL03|           1429811|
|   2026-01|    PL02|           1419252|
|   2026-01|    PL06|           1402185|
|   2026-01|    PL04|           1399488|
|   2026-01|    PL05|           1375887|
|   2026-02|    PL01|           1342273|
|   2026-02|    PL05|           1324271|
|   2026-02|    PL03|           1305531|
|   2026-02|    PL04|           1292557|
|   2026-02|    PL02|           1278581|
|   2026-02|    PL06|           1267678|
|   2026-03|    PL04|           1447603|
|   2026-03|    PL01|           1442140|
|   2026-03|    PL03|           1416484|
|   2026-03|    PL05|           1411435|
|   2026-03|    PL06|           1363719|
|   2026-03|    PL02|           1359740|
|   2026-04|    PL06|           1418607|
|   2026-04|    PL04|           1374631|
|   2026-04|    PL05|           1372000|
|   2026-04|    

In [179]:
date_parts_df.filter(
    (F.col("production_date") >= F.lit("2026-08-01").cast("date")) &
    (F.col("production_date") <= F.lit("2026-08-31").cast("date"))
).show(20, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+------------------+-------------+--------------+---------------+----------------+----------+-----------------------+----+-----+---+----------+---------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh        |temperature_c|vibration_mm_s|quality_status |maintenance_flag|supervisor|corrected_temperature_c|year|month|day|month_name|day_name |year_month|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+------------------+-------------+--------------+---------------+----------------+----------+-----------------------+----+-----+---+----------+---------+----------+
|PR0000001    |2026-08-07

In [180]:
date_parts_df.withColumn("date_plus_7_days", F.date_add("production_date", 7)).select(
    "production_id", "production_date", "date_plus_7_days"
).show(10)

+-------------+---------------+----------------+
|production_id|production_date|date_plus_7_days|
+-------------+---------------+----------------+
|    PR0000001|     2026-08-07|      2026-08-14|
|    PR0000002|     2026-08-03|      2026-08-10|
|    PR0000003|     2026-05-20|      2026-05-27|
|    PR0000004|     2026-01-15|      2026-01-22|
|    PR0000005|     2026-09-21|      2026-09-28|
|    PR0000006|     2026-09-03|      2026-09-10|
|    PR0000007|     2026-05-07|      2026-05-14|
|    PR0000008|     2026-08-17|      2026-08-24|
|    PR0000009|     2026-03-10|      2026-03-17|
|    PR0000010|     2026-02-13|      2026-02-20|
+-------------+---------------+----------------+
only showing top 10 rows


In [181]:
date_parts_df.withColumn("date_minus_30_days", F.date_sub("production_date", 30)).select(
    "production_id", "production_date", "date_minus_30_days"
).show(10)

+-------------+---------------+------------------+
|production_id|production_date|date_minus_30_days|
+-------------+---------------+------------------+
|    PR0000001|     2026-08-07|        2026-07-08|
|    PR0000002|     2026-08-03|        2026-07-04|
|    PR0000003|     2026-05-20|        2026-04-20|
|    PR0000004|     2026-01-15|        2025-12-16|
|    PR0000005|     2026-09-21|        2026-08-22|
|    PR0000006|     2026-09-03|        2026-08-04|
|    PR0000007|     2026-05-07|        2026-04-07|
|    PR0000008|     2026-08-17|        2026-07-18|
|    PR0000009|     2026-03-10|        2026-02-08|
|    PR0000010|     2026-02-13|        2026-01-14|
+-------------+---------------+------------------+
only showing top 10 rows


In [182]:
date_parts_df.withColumn(
    "days_from_current_date",
    F.datediff(F.current_date(), F.col("production_date"))
).select("production_id", "production_date", "days_from_current_date").show(10)

+-------------+---------------+----------------------+
|production_id|production_date|days_from_current_date|
+-------------+---------------+----------------------+
|    PR0000001|     2026-08-07|                    77|
|    PR0000002|     2026-08-03|                    81|
|    PR0000003|     2026-05-20|                   156|
|    PR0000004|     2026-01-15|                   281|
|    PR0000005|     2026-09-21|                    32|
|    PR0000006|     2026-09-03|                    50|
|    PR0000007|     2026-05-07|                   169|
|    PR0000008|     2026-08-17|                    67|
|    PR0000009|     2026-03-10|                   227|
|    PR0000010|     2026-02-13|                   252|
+-------------+---------------+----------------------+
only showing top 10 rows


In [183]:
monthly_output = date_parts_df.groupBy("year_month").agg(
    F.sum("units_produced").alias("total_output")
)
monthly_output.orderBy(F.col("total_output").desc()).show(1)

+----------+------------+
|year_month|total_output|
+----------+------------+
|   2026-08|     8529150|
+----------+------------+
only showing top 1 row
